# Localization failure investigation

Reproduce the source-quality and missed-event checks using local normalized outputs. Run `study-localization` first as documented in `docs/localization-evaluation.md`. This notebook reads no robot-control interface and changes no input data.

Source: [TUHH Robot Localization Failure Prediction Dataset](https://doi.org/10.15480/882.15836), CC BY 4.0. All results concern simulation data.


In [1]:
import json
from pathlib import Path

import polars as pl

from ros_telemetry_analytics.localization_eval import LocalizationEvalConfig
from ros_telemetry_analytics.localization_study import localization_diagnostics

root = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "configs/localization_study.json").is_file()
)
study_dir = root / "data/evaluations/localization-study"
study = json.loads((study_dir / "study.json").read_text())
print("Runs:", len(study["runs"]))
print("Samples:", sum(r["baseline"]["sample_count"] for r in study["runs"]))
print("Parquet failure labels:", sum(r["baseline"]["failure_sample_count"] for r in study["runs"]))

Runs: 21
Samples: 417185
Parquet failure labels: 96310


## Data quality

Record metrics retain upstream duplicates. Conflicting-label time intervals are excluded only from labeled-duration metrics; this avoids choosing a favorable label silently. CSV count discrepancies remain visible.


In [2]:
for key in study["runs"][0]["baseline"]["data_quality"]:
    print(key, sum(r["baseline"]["data_quality"][key] for r in study["runs"]))
print(
    "Excluded ambiguous seconds:",
    sum(r["baseline"]["time_metrics"]["excluded_ambiguous_duration_s"] for r in study["runs"]),
)
for run in study["runs"]:
    check = run["source_count_check"]
    if check["failure_sample_delta"]:
        print(
            run["run_id"],
            "CSV",
            check["declared_failure_samples"],
            "Parquet",
            check["observed_failure_samples"],
        )

conflicting_label_timestamp_groups 495
duplicate_timestamp_extra_records 112435
missing_both_signal_records 0
Excluded ambiguous seconds: 9.383333556
rec_20250821_132354 CSV 2776 Parquet 2778
rec_20250822_135308 CSV 8275 Parquet 8271
rec_20250828_105714 CSV 4860 Parquet 4861
rec_20250828_161448 CSV 6182 Parquet 6183
rec_20250825_135829 CSV 7306 Parquet 7309
rec_20250829_104641 CSV 3790 Parquet 3789
rec_20250825_161940 CSV 6513 Parquet 6511
rec_20250825_153216 CSV 8505 Parquet 8500


## Recompute the original warehouse baseline

This checks the stored normalized evidence against the published 15,708-sample baseline and separates uncovered failures from one-to-one matching limitations.


In [3]:
samples = pl.read_parquet(study_dir / "rec_20250821_104113.samples.parquet")
result = localization_diagnostics(samples, LocalizationEvalConfig())
assert samples.height == 15708
assert result["sample_metrics"]["true_positive"] == 2124
assert result["sample_metrics"]["false_negative"] == 2413
assert result["event_metrics"]["matched_event_count"] == 16
print(json.dumps(result["sample_metrics"], indent=2))
for row in result["failure_details"]:
    if not row["detected"]:
        print(
            row["expected_event_id"],
            row["classification"],
            f"{row['alerted_failure_records']}/{row['failure_records']}",
            "max spread",
            round(row["max_particle_spread_m"], 3),
        )

{
  "true_positive": 2124,
  "false_positive": 357,
  "false_negative": 2413,
  "true_negative": 10814,
  "precision": 0.8561064087061668,
  "recall": 0.4681507604143707,
  "f1": 0.6053006554573953
}
rec_20250821_104113:expected:5 unmatched_with_alert_coverage 247/247 max spread 0.543
rec_20250821_104113:expected:6 unmatched_with_alert_coverage 52/52 max spread 0.452
rec_20250821_104113:expected:8 no_alert_coverage 0/110 max spread 0.3
rec_20250821_104113:expected:10 no_alert_coverage 0/608 max spread 0.348
rec_20250821_104113:expected:19 no_alert_coverage 0/83 max spread 0.304
rec_20250821_104113:expected:21 no_alert_coverage 0/152 max spread 0.265
rec_20250821_104113:expected:23 no_alert_coverage 0/159 max spread 0.293
rec_20250821_104113:expected:24 no_alert_coverage 0/466 max spread 0.374


## Evaluation boundary

The environment split keeps all members and obstacle configurations together. The warehouse is development-only. Candidate selection uses development macro F1 with fixed precision and event-recall guardrails. `selection.json` is written before held-out runs are loaded. Detection-delay summaries exclude misses; false alarms/hour uses all observed segment time and is not a production alarm-rate estimate.
